# Tier 1: Computational & Physical Foundations
## Lab 1.0 — Foundations of Computational Structural Biology: UNIX, Python Data Structures, and Biomolecular Record Parsing

Welcome to **Lab 1.0** of the *Computational Structural Biology and Generative Macromolecular Design* curriculum! 

### Course & Audience Overview
This course is designed for junior and senior undergraduate students in computational biology, biochemistry, biophysics, bioengineering, and data science. Computational structural biology bridges physical chemistry, stereochemistry, statistical mechanics, and machine learning. To simulate or design biomacromolecules, we must first master the software environments and file standards where structural data lives.

### Why This Lab Matters
Before training diffusion models or running molecular dynamics simulations on GPUs, every structural bioinformatician must know how to:
1. Navigate Linux shell environments and leverage POSIX streams (`grep`, `awk`, `sed`, pipelines).
2. Write clean, vectorized Python code using core data structures (`dict`, `list`, `set`, `tuple`, generators).
3. Directly parse and manipulate the fundamental file format of structural biology: the **Protein Data Bank (PDB)** coordinate specification.
4. Compute fundamental physical properties—such as formal charges at physiological pH ($7.4$)—directly from raw atomic records without relying on black-box software.

---

### Learning Objectives
By the end of this laboratory, you will be able to:
- Navigate the Google Colab virtual machine file system using bash commands and IPython magics (`!`, `%%bash`, `%cd`).
- Programmatically download raw crystallographic structures from the RCSB Protein Data Bank over secure HTTPS protocols.
- Stream, filter, and inspect PDB coordinate records (`ATOM`, `HETATM`, `TER`, altLocs) using shell pipes and awk.
- Build Python classes and dictionary mappings to parse multi-chain crystallographic assemblies.
- Calculate the net charge of individual protein chains at pH $7.4$ using the Henderson-Hasselbalch equation and standard $\mathrm{p}K_a$ values.
- Self-assess your mastery using interactive embedded `jupyterquiz` questions that provide immediate, automated diagnostic feedback.

---
### System Architecture: Running System for Module 1
Across Module 1, we follow two continuous biological threads:
1. **The Protein Thread:** *Ideonella sakaiensis* **PETase** (polyethylene terephthalate hydrolase, PDB: `6ANE`), a bacterial enzyme engineered to depolymerize post-consumer plastic waste.
2. **The RNA Thread:** The *Vibrio vulnificus* *add* **A-riboswitch** (PDB: `1Y26`, Rfam: `RF00167`), an adenine-sensing regulatory RNA aptamer.

Let's begin by preparing our Google Colab computational environment.


### Section 0: Environment Setup and Dependency Management

Google Colab instances run on an Ubuntu Linux container. In this cell, we install the required Python libraries using `pip`:
- `biopython`: Industrial-standard library for biological computation, sequence handling, and structure parsing.
- `jupyterquiz`: Interactive self-testing widget for immediate formative assessment with automated explanations.
- `py3Dmol`: WebGL-based molecular visualizer designed specifically for Jupyter/Colab notebooks.
- `pandas` & `numpy`: High-performance vectorized numerical computation and structured data analysis.

> **Colab Tip:** The `!` operator executes a command in the underlying Linux shell rather than the Python interpreter. The `-q` flag ensures a quiet installation without overwhelming log outputs.


In [ ]:
# Install pinned, stable dependencies (including valid jupyterquiz and gemmi)
!pip install -q biopython==1.83 jupyterquiz==2.8.1 py3Dmol==2.1.0 gemmi==0.6.5 pandas numpy matplotlib requests

import sys
import os
import random
import numpy as np
import pandas as pd
import requests
import gemmi
from IPython.display import display

# Fix global random seed for computational reproducibility across student runs
COURSE_SEED = 42
random.seed(COURSE_SEED)
np.random.seed(COURSE_SEED)

print("Environment initialized successfully.")
print(f"Python version: {sys.version.split()[0]}")
print(f"Gemmi version: {gemmi.__version__}")
print(f"Reproducibility seed set to: {COURSE_SEED}")


---
## Section 1: The Linux Shell & Stream Processing for Structural Biology

Biological coordinate files can be massive, often containing hundreds of thousands of lines. Graphical text editors frequently crash or lag when opening large crystal structures or molecular dynamics trajectory frames. The POSIX command-line interface provides high-throughput stream-processing tools:
- `pwd`: Print Working Directory.
- `ls -lh`: List files with human-readable file sizes.
- `mkdir`: Make directory.
- `curl` / `wget`: Non-interactive network downloaders.
- `grep`: Global Regular Expression Print (scans input lines matching a pattern).
- `awk`: Pattern-scanning and field-processing programming language.
- `wc -l`: Word count utility (when passed `-l`, counts newline characters).
- `|` (Pipe): Streams `stdout` from one process directly into `stdin` of the next process without intermediate disk I/O.

Let's programmatically retrieve the crystal structure of *Ideonella sakaiensis* PETase (PDB ID: **`6ANE`**) directly from the RCSB Protein Data Bank.


In [ ]:
# Create a dedicated directory for our structural data
os.makedirs("data", exist_ok=True)

# Fetch 6ANE coordinate file from RCSB via HTTPS
pdb_id = "6ane"
url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
dest_path = os.path.join("data", f"{pdb_id}.pdb")

if not os.path.exists(dest_path):
    print(f"Downloading {pdb_id.upper()} from {url}...")
    response = requests.get(url)
    response.raise_for_status()
    with open(dest_path, "w") as f:
        f.write(response.text)
    print(f"File successfully written to {dest_path}")
else:
    print(f"File {dest_path} already exists locally.")


### Anatomical Inspection of a PDB File via Shell Streams

A `.pdb` file is an 80-column fixed-width ASCII record stream. The primary coordinate records are:
1. **`ATOM`**: Standard amino acid or nucleic acid polymer atoms.
2. **`HETATM`** (Hetero-atom): Non-polymer entities, including co-crystallized small-molecule ligands, cofactor ions ($\\mathrm{Mg}^{2+}$, $\\mathrm{Zn}^{2+}$), buffer salts, and crystallographic solvent waters ($\\mathrm{HOH}$).
3. **`TER`**: Terminal record indicating the covalent termination of a polypeptide or nucleotide chain.
4. **`ANISOU`**: Anisotropic displacement parameters.

Let's use shell pipes (`!grep ... | wc -l`) to query the raw counts of atoms, alpha-carbons (`CA`), and solvent waters without loading the file into Python memory.


In [ ]:
# 1. Total number of lines in the PDB file
!wc -l data/6ane.pdb

# 2. Total ATOM records (macromolecular polymer atoms)
!grep "^ATOM  " data/6ane.pdb | wc -l

# 3. Total HETATM records (ligands, ions, waters)
!grep "^HETATM" data/6ane.pdb | wc -l

# 4. Total alpha-carbon (CA) atoms across all chains (each standard residue has exactly one CA)
!grep "^ATOM  " data/6ane.pdb | grep " CA " | wc -l

# 5. Total crystallographic water oxygen atoms (resname HOH)
!grep "^HETATM" data/6ane.pdb | grep "HOH" | wc -l


### Column Slicing with `awk`
Because classic PDB format relies on strict column positions (rather than tabs or commas), `awk` allows us to extract metadata rapidly.

Let's inspect the first 10 alpha-carbon records of Chain A:
- Column 1: Record name (`ATOM`)
- Column 2: Atom serial number
- Column 3: Atom name (`CA`)
- Column 4: Residue name (`GLN`, `THR`, etc.)
- Column 5: Chain identifier (`A`, `B`, `C`, etc.)
- Column 6: Residue sequence number
- Columns 7, 8, 9: Cartesian coordinates $X, Y, Z$ in Ångströms ($\mathrm{\AA}$)
- Column 10: Occupancy (fraction of unit cells with this atom in this position, usually $1.00$)
- Column 11: Temperature factor ($B$-factor, measuring atomic positional displacement in $\mathrm{\AA}^2$)


In [ ]:
# Extract Atom Index, Residue Name, Residue Number, and B-Factor for the first 10 CA atoms in Chain A
!awk '$1=="ATOM" && $3=="CA" && $5=="A" {printf "%-6s Residue: %3s %-4d | Coord: (%7.2f, %7.2f, %7.2f) | B-factor: %5.2f\n", $3, $4, $6, $7, $8, $9, $11}' data/6ane.pdb | head -n 10


---
### Section 1.5: The Modern Standard — Transitioning from Legacy PDB to PDBx/mmCIF

While legacy `.pdb` fixed-width format is historically foundational, the worldwide Protein Data Bank (wwPDB) designated **PDBx/mmCIF (macromolecular Crystallographic Information File)** as the official archive standard.

#### Why Legacy PDB Broke Down:
1. **Atom Serial Limitation:** Columns 7–11 allow only 5 digits, capping files at **99,999 atoms**.
2. **Chain Identifier Limitation:** Column 22 permits only a **single character** (A–Z, a–z, 0–9), capping complexes at **62 chains**.
3. **Massive Cryo-EM & Ribosomal Assemblies:** Ribosomes, vault nanoparticles, and viral capsids easily exceed 500,000 atoms and hundreds of chains, requiring split PDB files.

In contrast, `.cif` uses relational table loops (`_atom_site.*`) with unbounded arbitrary string widths and multi-character auth/label chain IDs.

Let's programmatically retrieve the modern `.cif` coordinate file for PETase (`6ane.cif`) and inspect it with **`gemmi`**.


In [ ]:
# Download modern mmCIF coordinate file from RCSB
pdb_id = globals().get('pdb_id', '6ane')
cif_url = f"https://files.rcsb.org/download/{pdb_id}.cif"
cif_path = os.path.join("data", f"{pdb_id}.cif")

if not os.path.exists(cif_path):
    res = requests.get(cif_url)
    res.raise_for_status()
    with open(cif_path, "w") as f:
        f.write(res.text)
    print(f"Downloaded modern mmCIF file to {cif_path}")
else:
    print(f"{cif_path} already exists locally.")

# Parse using high-performance gemmi library
doc = gemmi.cif.read_file(cif_path)
block = doc.sole_block()

# Query experimental resolution and R-factors directly from CIF tags
res_val = block.find_value('_refine.ls_d_res_high')
r_work_val = block.find_value('_refine.ls_R_factor_R_work')
r_free_val = block.find_value('_refine.ls_R_factor_R_free')

print(f"mmCIF Header Validation via Gemmi:")
print(f"  Resolution (_refine.ls_d_res_high): {res_val} Å")
print(f"  R_work: {r_work_val} | R_free: {r_free_val}")

# Access structural hierarchy natively with gemmi
cif_structure = gemmi.read_structure(cif_path)
print(f"  Total Models: {len(cif_structure)} | Chains in Model 0: {[ch.name for ch in cif_structure[0]]}")


### Formative Self-Assessment: Shell Literacy & PDB Records
Test your understanding of PDB record architecture and stream processing before moving into Python data structures.


In [ ]:
from jupyterquiz import display_quiz

quiz_section_1 = [
    {
        "question": "What is the key structural distinction between an ATOM record and a HETATM record in a standard PDB file?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "ATOM records denote standard amino acid or nucleotide polymer atoms, whereas HETATM denotes non-polymer entities such as ligands, cofactor ions, and crystallographic water molecules.",
                "correct": True,
                "feedback": "Correct! ATOM defines the primary biological polymers (protein/nucleic acid backbones and side chains), while HETATM holds heteroatoms, cofactors, and water."
            },
            {
                "answer": "ATOM records are experimentally determined coordinates, while HETATM records are computationally modeled or predicted coordinates.",
                "correct": False,
                "feedback": "Incorrect. Both ATOM and HETATM records in crystal structures derive from experimental electron density maps."
            },
            {
                "answer": "ATOM records represent heavy atoms (carbon, nitrogen, oxygen), while HETATM records are reserved solely for hydrogen atoms.",
                "correct": False,
                "feedback": "Incorrect. Hydrogen atoms belonging to amino acids are labeled as ATOM records, not HETATM."
            },
            {
                "answer": "ATOM records indicate alpha-helices, while HETATM indicates beta-sheets.",
                "correct": False,
                "feedback": "Incorrect. Secondary structure classifications are annotated in HELIX and SHEET header records, not ATOM/HETATM."
            }
        ]
    },
    {
        "question": "In the Unix command `grep \"^ATOM  \" 6ane.pdb | grep \" CA \" | wc -l`, what does the final numeric output physically represent for a protein?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "The total number of amino acid residues with resolved alpha-carbon atoms across all chains in the structure.",
                "correct": True,
                "feedback": "Correct! Because every standard amino acid has exactly one alpha-carbon (CA), counting the CA atoms yields the total resolved amino acid count."
            },
            {
                "answer": "The total molecular weight of the protein in kilodaltons.",
                "correct": False,
                "feedback": "Incorrect. 'wc -l' counts newline characters (records), not molecular weights."
            },
            {
                "answer": "The number of calcium (Ca) ions bound to the catalytic cleft.",
                "correct": False,
                "feedback": "Incorrect. Calcium ions would be denoted as HETATM CA (or CA ion) in the heteroatom section, whereas grep \"^ATOM  \" restricts the query solely to polymer atoms."
            },
            {
                "answer": "The number of beta-strands in the catalytic domain.",
                "correct": False,
                "feedback": "Incorrect. Beta-strands are collections of multiple amino acid residues, not single CA records."
            }
        ]
    }
]

display_quiz(quiz_section_1)


---
## Section 2: Python Data Structures for Structural Biology

While shell pipelines are fast for simple file inspections, complex geometrical and physical calculations require parsing data into object-oriented Python representations.

Let's build a lightweight, robust, pure-Python atomic parser. We will define an `Atom` class and a `Residue` container that parses fixed-width columns directly without relying on external dependencies.


In [ ]:
class Atom:
    """Represents a single atomic coordinate record from a PDB file."""
    __slots__ = ('record_type', 'serial', 'name', 'alt_loc', 'res_name', 
                 'chain_id', 'res_seq', 'x', 'y', 'z', 'occupancy', 'b_factor', 'element')
    
    def __init__(self, line: str):
        # Strict PDB format column slicing (1-indexed specifications):
        # 1-6: Record name | 7-11: Atom serial | 13-16: Atom name | 17: AltLoc
        # 18-20: ResName | 22: ChainID | 23-26: ResSeq | 31-38: X | 39-46: Y | 47-54: Z
        # 55-60: Occupancy | 61-66: Temperature factor | 77-78: Element symbol
        self.record_type = line[0:6].strip()
        self.serial = int(line[6:11].strip())
        self.name = line[12:16].strip()
        self.alt_loc = line[16].strip()
        self.res_name = line[17:20].strip()
        self.chain_id = line[21].strip()
        self.res_seq = int(line[22:26].strip())
        self.x = float(line[30:38].strip())
        self.y = float(line[38:46].strip())
        self.z = float(line[46:54].strip())
        self.occupancy = float(line[54:60].strip()) if line[54:60].strip() else 1.0
        self.b_factor = float(line[60:66].strip()) if line[60:66].strip() else 0.0
        self.element = line[76:78].strip() if len(line) >= 78 else self.name[0]

    @property
    def coords(self) -> np.ndarray:
        return np.array([self.x, self.y, self.z], dtype=np.float64)

    def __repr__(self):
        return f"<Atom {self.name} (#{self.serial}) of {self.res_name}{self.res_seq}:{self.chain_id}>"

def parse_pdb_atoms(pdb_file_path: str):
    """Parses all ATOM and HETATM records into a structured Python list."""
    atoms = []
    with open(pdb_file_path, 'r') as fh:
        for line in fh:
            if line.startswith(('ATOM  ', 'HETATM')):
                try:
                    atoms.append(Atom(line))
                except (ValueError, IndexError):
                    continue
    return atoms

# Parse 6ANE
all_atoms = parse_pdb_atoms("data/6ane.pdb")
print(f"Parsed {len(all_atoms)} total atoms from 6ANE.")
print(f"Sample Atom: {all_atoms[0]}")
print(f"Coordinates: {all_atoms[0].coords} Å")


### Structural Inventory: Chains, Alternate Conformers, and Residues

In protein crystallography, some atoms or side chains adopt multiple conformations within the crystal lattice. These are flagged with an **Alternate Location Indicator** (`altLoc`, e.g., 'A', 'B'), with partial occupancies summing to $1.0$.

Let's group our parsed atoms into hierarchical dictionaries:
$$\text{Structure} \longrightarrow \text{Chains} \longrightarrow \text{Residues} \longrightarrow \text{Atoms}$$


In [ ]:
from collections import defaultdict

def build_structure_hierarchy(atoms):
    """
    Organizes a flat atom list into chains and residues.
    Filters out alternate conformers (keeping the primary conformer 'A' or blank ' ').
    """
    hierarchy = defaultdict(lambda: defaultdict(list))
    altloc_counter = defaultdict(set)
    
    for atom in atoms:
        # Filter: keep only standard polymer ATOM records
        if atom.record_type == "ATOM":
            # Track altLocs
            if atom.alt_loc:
                altloc_counter[(atom.chain_id, atom.res_seq)].add(atom.alt_loc)
            # Retain only primary conformer 'A' or unflagged conformer ' '
            if atom.alt_loc in ('', 'A', '1'):
                hierarchy[atom.chain_id][atom.res_seq].append(atom)
                
    return hierarchy, altloc_counter

chains_dict, altlocs = build_structure_hierarchy(all_atoms)

print(f"Detected chains in 6ANE: {list(chains_dict.keys())}")
for chain_id, residues in chains_dict.items():
    print(f"Chain {chain_id}: {len(residues)} amino acid residues resolved.")

print(f"\nTotal residues exhibiting conformational heterogeneity (altLocs): {len(altlocs)}")
first_few_alts = list(altlocs.items())[:5]
for (cid, rnum), conformers in first_few_alts:
    print(f"  Chain {cid}, Residue #{rnum} has alternate conformers: {sorted(list(conformers))}")


### Formative Self-Assessment: Python Hierarchy & AltLocs
Verify your understanding of crystallographic occupancies and data parsing.


In [ ]:
quiz_section_2 = [
    {
        "question": "If an atom in a crystal structure is assigned an occupancy of 0.60 with altLoc 'A', what does this physically signify?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "In approximately 60% of the unit cells throughout the crystal lattice, that atom occupies conformation 'A'.",
                "correct": True,
                "feedback": "Correct! Occupancy measures the fractional presence of that atomic coordinate across the ensemble of unit cells in the crystal."
            },
            {
                "answer": "The experimental resolution at that site is 0.60 Ångströms.",
                "correct": False,
                "feedback": "Incorrect. Resolution is a global diffraction metric (typically 1.5 - 3.0 Å), not a per-atom occupancy."
            },
            {
                "answer": "The atom has an electrostatic charge of +0.60 elementary charges.",
                "correct": False,
                "feedback": "Incorrect. Formal charges and partial charges are not recorded in the PDB occupancy column."
            },
            {
                "answer": "60% of the residue has been degraded or proteolyzed during crystallization.",
                "correct": False,
                "feedback": "Incorrect. Fractional occupancy denotes conformational heterogeneity in an intact polypeptide, not covalent degradation."
            }
        ]
    }
]

display_quiz(quiz_section_2)


---
## Section 3: Biomolecular Properties — Net Charge under the Physiological pH 7.4 Contract

In structural biology, the electrostatic charge of a protein controls its solubility, folding stability, active-site reactivity, and interactions with polyanionic nucleic acids (DNA/RNA).

### The Henderson-Hasselbalch Equation
The ionization state of each titratable side chain is governed by its acid dissociation constant, $\mathrm{p}K_a$. At an environmental $\mathrm{pH}$, the ratio of deprotonated to protonated species is given by:
$$\mathrm{pH} = \mathrm{p}K_a + \log_{10}\left(\frac{[\text{Base}]}{[\text{Acid}]}\right)$$

#### 1. Acidic (Anionic) Side Chains: Aspartate (Asp), Glutamate (Glu), Cysteine (Cys), Tyrosine (Tyr), and C-terminus
These groups are neutral when protonated ($\text{HA}$) and negatively charged ($-1$) when deprotonated ($\text{A}^-$):
$$q_{\text{acidic}} = -\frac{1}{1 + 10^{\mathrm{p}K_a - \mathrm{pH}}}$$

#### 2. Basic (Cationic) Side Chains: Lysine (Lys), Arginine (Arg), Histidine (His), and N-terminus
These groups are positively charged ($+1$) when protonated ($\text{BH}^+$) and neutral ($0$) when deprotonated ($\text{B}$):
$$q_{\text{basic}} = +\frac{1}{1 + 10^{\mathrm{pH} - \mathrm{p}K_a}}$$

### The Pinned Physiological pH 7.4 Protocol Contract
In this course, we adhere to a strict structural biology contract:
- Environmental $\mathrm{pH} = 7.40$.
- Standard amino-acid terminal groups: N-terminus ($\mathrm{p}K_a = 7.7$), C-terminus ($\mathrm{p}K_a = 3.3$).
- Side-chain $\mathrm{p}K_a$ reference values:
  - $\text{Asp}: 3.9$
  - $\text{Glu}: 4.1$
  - $\text{His}: 6.0$ *(Notice: at pH 7.4, Histidine is predominantly neutral unprotonated free base, carrying only $\approx +0.038$ partial charge! It is NOT a fixed $+1$ cation)*
  - $\text{Cys}: 8.3$
  - $\text{Tyr}: 10.5$
  - $\text{Lys}: 10.5$
  - $\text{Arg}: 12.5$

Let's implement the vectorized Henderson-Hasselbalch titration in Python and calculate the net charge of PETase Chain A.


In [ ]:
# Course-pinned Henderson-Hasselbalch pKa table
PKA_TABLE = {
    'N_TERM': 7.7,
    'C_TERM': 3.3,
    'ASP': 3.9,
    'GLU': 4.1,
    'HIS': 6.0,
    'CYS': 8.3,
    'TYR': 10.5,
    'LYS': 10.5,
    'ARG': 12.5
}

BASIC_RESIDUES = {'LYS', 'ARG', 'HIS'}
ACIDIC_RESIDUES = {'ASP', 'GLU', 'CYS', 'TYR'}

def calculate_residue_charge(res_name: str, pH: float = 7.4) -> float:
    """Calculates the fractional charge of a titratable residue at a given pH."""
    res = res_name.upper()
    if res in BASIC_RESIDUES:
        pka = PKA_TABLE[res]
        # Basic: +1 when protonated
        return 1.0 / (1.0 + 10.0 ** (pH - pka))
    elif res in ACIDIC_RESIDUES:
        pka = PKA_TABLE[res]
        # Acidic: -1 when deprotonated
        return -1.0 / (1.0 + 10.0 ** (pka - pH))
    return 0.0

def calculate_chain_net_charge(residue_list: list, pH: float = 7.4) -> float:
    """
    Computes total formal charge of a protein chain including termini.
    """
    # N-terminal amino group charge (+1 when protonated)
    total_charge = 1.0 / (1.0 + 10.0 ** (pH - PKA_TABLE['N_TERM']))
    
    # C-terminal carboxyl group charge (-1 when deprotonated)
    total_charge += -1.0 / (1.0 + 10.0 ** (PKA_TABLE['C_TERM'] - pH))
    
    # Sum side chain contributions
    for res_name in residue_list:
        total_charge += calculate_residue_charge(res_name, pH)
        
    return total_charge

# Extract primary sequence of 6ANE Chain A
chain_a_residues = [atoms[0].res_name for rseq, atoms in sorted(chains_dict['A'].items())]
net_q_74 = calculate_chain_net_charge(chain_a_residues, pH=7.4)

print(f"PETase Chain A sequence length: {len(chain_a_residues)} amino acids")
print(f"Calculated Net Charge at physiological pH 7.40: {net_q_74:+.3f} e")


### Complete Titration Curve & Isoelectric Point (pI)
The **isoelectric point** ($\mathrm{p}I$) is the exact $\mathrm{pH}$ at which the net electrical charge of the macromolecule equals zero ($q(\mathrm{p}I) = 0$). At this point, proteins typically exhibit their lowest aqueous solubility because intermolecular electrostatic repulsions are minimized.

Let's compute the titration curve across $\mathrm{pH} \in [1, 14]$ and determine $\mathrm{p}I$ via binary bisection search.


In [ ]:
import matplotlib.pyplot as plt

def compute_titration_curve(residue_list, pH_range=np.linspace(1.0, 14.0, 150)):
    charges = [calculate_chain_net_charge(residue_list, pH=p) for p in pH_range]
    return pH_range, np.array(charges)

def calculate_isoelectric_point(residue_list) -> float:
    """Finds pI using bisection search where net charge crosses zero."""
    low_pH, high_pH = 0.0, 14.0
    for _ in range(50):
        mid_pH = (low_pH + high_pH) / 2.0
        q = calculate_chain_net_charge(residue_list, pH=mid_pH)
        if q > 0:
            low_pH = mid_pH
        else:
            high_pH = mid_pH
    return (low_pH + high_pH) / 2.0

pH_grid, q_curve = compute_titration_curve(chain_a_residues)
pI = calculate_isoelectric_point(chain_a_residues)

print(f"Calculated Isoelectric Point (pI): {pI:.2f}")

# Plot titration curve
plt.figure(figsize=(7, 4), dpi=100)
plt.plot(pH_grid, q_curve, color='#1f77b4', lw=2.5, label='PETase Chain A Net Charge')
plt.axhline(0, color='gray', linestyle='--', lw=1.0)
plt.axvline(7.4, color='#d62728', linestyle=':', lw=1.5, label=f'pH 7.4 (q = {net_q_74:+.2f} e)')
plt.axvline(pI, color='#2ca02c', linestyle='-.', lw=1.5, label=f'pI = {pI:.2f}')
plt.title("Protein Electrostatic Titration Curve (Henderson-Hasselbalch)", fontsize=12, fontweight='bold')
plt.xlabel("pH", fontsize=11)
plt.ylabel("Net Charge (Elementary Charges, e)", fontsize=11)
plt.grid(True, alpha=0.3)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()


### Formative Self-Assessment: Electrostatic Titration
Test your understanding of side-chain protonation equilibria and net charge.


In [ ]:
quiz_section_3 = [
    {
        "question": "At physiological pH 7.40, why is Histidine (pKa ≈ 6.0) NOT considered a fully charged (+1) basic residue?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "Because pH (7.4) is significantly greater than its pKa (6.0), the equilibrium heavily favors the deprotonated, neutral imidazole form (~96% uncharged, ~4% cation).",
                "correct": True,
                "feedback": "Correct! By Henderson-Hasselbalch, when pH > pKa for a base, it is predominantly deprotonated and uncharged."
            },
            {
                "answer": "Because Histidine is an aliphatic hydrophobic amino acid that remains buried in the protein core.",
                "correct": False,
                "feedback": "Incorrect. Histidine has an ionizable polar/aromatic imidazole ring."
            },
            {
                "answer": "Because Histidine forms covalent peptide bonds that eliminate its side-chain charge.",
                "correct": False,
                "feedback": "Incorrect. Peptide bonds involve the alpha-amino and alpha-carboxyl groups, leaving the imidazole side chain free to titrate."
            },
            {
                "answer": "Because Histidine carries a -1 charge at pH 7.4.",
                "correct": False,
                "feedback": "Incorrect. Basic groups transition between 0 (deprotonated) and +1 (protonated); they are never -1."
            }
        ]
    },
    {
        "question": "What is the physical definition of the isoelectric point (pI) of a protein?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "The pH at which the total sum of positive and negative formal charges on the protein equals zero.",
                "correct": True,
                "feedback": "Correct! At pH = pI, net charge is zero."
            },
            {
                "answer": "The temperature at which the protein unfolds and loses secondary structure.",
                "correct": False,
                "feedback": "Incorrect. That defines the melting temperature (Tm)."
            },
            {
                "answer": "The pH at which all titratable residues are 100% protonated.",
                "correct": False,
                "feedback": "Incorrect. That occurs only at extremely acidic pH (< 1)."
            },
            {
                "answer": "The salt concentration required to precipitate the protein out of solution.",
                "correct": False,
                "feedback": "Incorrect. That relates to salting out / ionic strength."
            }
        ]
    }
]

display_quiz(quiz_section_3)


---
## Section 4: 3D Molecular Graphics in the Browser with py3Dmol

A 3D coordinate model is a structural hypothesis derived from experimental observations (X-ray diffraction, cryo-EM, or NMR).

To understand molecular function, we visualize models using standard graphical representations:
- **Cartoon:** Traces the secondary structure backbone (alpha-helices as spirals, beta-strands as arrows pointing $5' \to 3'$ or $\mathrm{N} \to \mathrm{C}$, and loops as smooth tubes).
- **Sticks:** Displays atomic bonds as cylinders with CPK element coloring (Carbon = cyan/gray, Nitrogen = blue, Oxygen = red, Sulfur = yellow).
- **Spheres (van der Waals):** Displays steric space-filling volume.

### The PETase Catalytic Triad & Numbering Discipline
*Ideonella sakaiensis* PETase is a serine hydrolase operating through a classic **Ser-His-Asp catalytic triad**:
- **Nucleophile:** $\text{Ser160}$ (mature sequence numbering)
- **General Acid/Base:** $\text{His237}$
- **Electrostatic Anchor:** $\text{Asp206}$
- **Substrate-Wobbling Aromatic:** $\text{Trp185}$ (forms pi-stacking interactions with the terephthalate ring)

> **Crucial Numbering Protocol:**  
> In the deposited PDB `6ANE` coordinate file, the coordinates carry an offset of $-27$ relative to the mature functional sequence (because residues 1–27 comprise the cleaved signal peptide):  
> $\text{Mature Residue} = \text{Deposited Residue} + 27$  
> Therefore, in the raw coordinate file:
> - $\text{Ser160} \implies \text{PDB } \mathbf{133}$
> - $\text{Asp206} \implies \text{PDB } \mathbf{179}$
> - $\text{His237} \implies \text{PDB } \mathbf{210}$
> - $\text{Trp185} \implies \text{PDB } \mathbf{158}$

Let's render PETase in 3D using `py3Dmol`, highlighting this catalytic machinery.


In [ ]:
import py3Dmol

def render_petase_active_site(pdb_path: str):
    """Renders 6ANE with active site triad and Trp highlighted as sticks."""
    view = py3Dmol.view(width=700, height=450)
    with open(pdb_path, 'r') as f:
        pdb_data = f.read()
    
    view.addModel(pdb_data, "pdb")
    
    # Base style: semi-transparent cartoon
    view.setStyle({}, {"cartoon": {"color": "#e0e0e0", "opacity": 0.85}})
    
    # Active site residues (using deposited PDB sequence numbering: mature - 27)
    # Ser160 -> 133 | Asp206 -> 179 | His237 -> 210 | Trp185 -> 158
    catalytic_selection = {"resi": ["133", "179", "210"], "chain": "A"}
    wobble_selection = {"resi": ["158"], "chain": "A"}
    
    # Highlight catalytic triad as cyan sticks
    view.addStyle(catalytic_selection, {"stick": {"colorscheme": "cyanCarbon", "radius": 0.25}})
    # Highlight Trp185 aromatic clamp as magenta sticks
    view.addStyle(wobble_selection, {"stick": {"colorscheme": "magentaCarbon", "radius": 0.25}})
    
    # Add textual residue labels
    view.addResLabels({"resi": "133", "chain": "A"}, {"fontColor": "black", "fontSize": 12, "showBackground": True, "backgroundOpacity": 0.7})
    view.addResLabels({"resi": "179", "chain": "A"}, {"fontColor": "black", "fontSize": 12, "showBackground": True, "backgroundOpacity": 0.7})
    view.addResLabels({"resi": "210", "chain": "A"}, {"fontColor": "black", "fontSize": 12, "showBackground": True, "backgroundOpacity": 0.7})
    view.addResLabels({"resi": "158", "chain": "A"}, {"fontColor": "black", "fontSize": 12, "showBackground": True, "backgroundOpacity": 0.7})
    
    # Center camera on the active site nucleophile Ser133 (mature Ser160)
    view.zoomTo({"resi": "133", "chain": "A"})
    return view

# Render the active site view
viewer = render_petase_active_site("data/6ane.pdb")
viewer.show()


### Accessible Tabular Residue Summary
Per our universal scientific accessibility standards, every 3D figure is accompanied by an explicit tabular summary of all highlighted functional residues.


In [ ]:
active_site_table = pd.DataFrame([
    {"Chain": "A", "Mature Res #": 160, "Deposited PDB #": 133, "Residue": "SER", "Chemical Role": "Catalytic nucleophile (attacks PET ester carbonyl)"},
    {"Chain": "A", "Mature Res #": 206, "Deposited PDB #": 179, "Residue": "ASP", "Chemical Role": "Electrostatic triad base; stabilizes protonated His"},
    {"Chain": "A", "Mature Res #": 237, "Deposited PDB #": 210, "Residue": "HIS", "Chemical Role": "General acid/base; activates Ser nucleophile"},
    {"Chain": "A", "Mature Res #": 185, "Deposited PDB #": 158, "Residue": "TRP", "Chemical Role": "Aromatic clamp (pi-stacking with terephthalate ring)"}
])

print("Table 1.1: Functional Residue Mapping for PETase (PDB: 6ANE)")
display(active_site_table)


---
## Section 5: The Nucleic Acid Thread — Purine Riboswitch (PDB: 1Y26)

Riboswitches are structured non-coding RNA elements residing in the $5'$-untranslated regions ($5'$-UTRs) of messenger RNAs that bind specific metabolites, modulating transcription or translation.

Unlike proteins, which possess diverse uncharged, basic, and acidic residues:
1. The RNA backbone is **polyanionic**: each nucleotide monophosphate linkage carries a net charge of approximately $-1.0\,e$ at physiological pH $7.4$.
2. RNA folding is driven by **Watson-Crick base pairing** (G-C has 3 hydrogen bonds; A-U has 2 hydrogen bonds), **base stacking**, and coordination of divalent cations (such as $\mathrm{Mg}^{2+}$) that shield the mutually repulsive negative charges of the phosphodiester backbone.

Let's fetch and render the *Vibrio vulnificus* *add* **A-riboswitch** (PDB: `1Y26`), highlighting its bound adenine ligand (`ADE`) and the critical **U74** nucleotide that confers molecular selectivity.


In [ ]:
# Download 1Y26 RNA crystal structure from RCSB
rna_pdb_id = "1y26"
rna_url = f"https://files.rcsb.org/download/{rna_pdb_id}.pdb"
rna_dest = os.path.join("data", f"{rna_pdb_id}.pdb")

if not os.path.exists(rna_dest):
    print(f"Downloading RNA structure {rna_pdb_id.upper()}...")
    res = requests.get(rna_url)
    res.raise_for_status()
    with open(rna_dest, "w") as f:
        f.write(res.text)
    print("Download complete.")
else:
    print(f"{rna_dest} already exists locally.")

def render_riboswitch_pocket(rna_path: str):
    """Visualizes 1Y26 RNA aptamer with adenine ligand and U74 discriminator."""
    v = py3Dmol.view(width=700, height=450)
    with open(rna_path, 'r') as f:
        rna_data = f.read()
    v.addModel(rna_data, "pdb")
    
    # Render RNA cartoon colored 5' -> 3' by residue spectrum
    v.setStyle({}, {"cartoon": {"color": "spectrum", "opacity": 0.8}})
    
    # Highlight bound Adenine metabolite (HETATM record: resname ADE) as yellow sticks
    v.addStyle({"resn": "ADE"}, {"stick": {"colorscheme": "yellowCarbon", "radius": 0.25}})
    
    # Highlight U74 discriminator nucleotide as green sticks
    v.addStyle({"resi": "74"}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.25}})
    
    # Add labels
    v.addResLabels({"resn": "ADE"}, {"fontColor": "black", "fontSize": 12, "showBackground": True, "backgroundOpacity": 0.7})
    v.addResLabels({"resi": "74"}, {"fontColor": "black", "fontSize": 12, "showBackground": True, "backgroundOpacity": 0.7})
    
    # Zoom to the ligand-binding pocket
    v.zoomTo({"resn": "ADE"})
    return v

rna_viewer = render_riboswitch_pocket("data/1y26.pdb")
rna_viewer.show()


### Accessible Tabular Summary of RNA Ligand Binding
Below is the accessible summary table detailing the critical binding pocket interactions of the purine riboswitch.


In [ ]:
rna_table = pd.DataFrame([
    {"Chain": "X", "Residue": "ADE", "Index": "HETATM", "Functional Role": "Bound adenine ligand (metabolic signal)"},
    {"Chain": "X", "Residue": "U", "Index": 74, "Functional Role": "Watson-Crick discriminator base; forms canonical base-pair with adenine"},
    {"Chain": "X", "Residue": "U", "Index": 51, "Functional Role": "Aptamer binding pocket roof; base-stacking stabilization"},
    {"Chain": "X", "Residue": "U", "Index": 22, "Functional Role": "Core tertiary junction loop; hydrogen bonding to ligand"}
])

print("Table 1.2: Purine Riboswitch Binding Pocket Architecture (PDB: 1Y26)")
display(rna_table)


### Formative Self-Assessment: Catalytic Triads & RNA Pockets
Test your comprehension of the 3D active sites of PETase and the purine riboswitch.


In [ ]:
quiz_section_4 = [
    {
        "question": "Why is Trp185 in PETase termed a 'wobbling' aromatic clamp, and why is its side chain critical for breaking down PET plastic?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "It provides hydrophobic pi-stacking interactions with the aromatic terephthalate ring of the polymer and displays alternate conformers in crystal structures reflecting mobility during catalysis.",
                "correct": True,
                "feedback": "Correct! Trp185 stacks against the PET polymer benzene ring and exhibits crystallographic mobility between open and closed conformations."
            },
            {
                "answer": "It acts as the nucleophile that directly cleaves the ester bond.",
                "correct": False,
                "feedback": "Incorrect. Ser160 (deposited Ser133) is the catalytic nucleophile."
            },
            {
                "answer": "It binds a catalytic zinc ion required for water deprotonation.",
                "correct": False,
                "feedback": "Incorrect. PETase is a serine hydrolase, not a metalloprotease."
            },
            {
                "answer": "It forms Watson-Crick hydrogen bonds with the incoming mRNA substrate.",
                "correct": False,
                "feedback": "Incorrect. PETase acts on synthetic polyethylene terephthalate polymer, not mRNA."
            }
        ]
    },
    {
        "question": "What is the primary architectural factor that renders the RNA phosphodiester backbone polyanionic at pH 7.4?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "Each phosphodiester diester linkage has an ionizable OH with a pKa < 2, remaining virtually 100% deprotonated with a net formal charge of -1 e per nucleotide.",
                "correct": True,
                "feedback": "Correct! Phosphates are strong acids (pKa ~ 1-2); at neutral pH, each internal backbone phosphate carries -1 formal charge."
            },
            {
                "answer": "Ribose sugars lose their 2'-OH protons at neutral pH to become anionic alkoxides.",
                "correct": False,
                "feedback": "Incorrect. Ribose 2'-OH hydroxyls have pKa values > 12 and remain neutral at pH 7.4."
            },
            {
                "answer": "The nitrogenous bases (A, U, G, C) are all negatively charged at pH 7.4.",
                "correct": False,
                "feedback": "Incorrect. Purines and pyrimidines are neutral uncharged aromatic bases at pH 7.4."
            },
            {
                "answer": "Magnesium ions covalently attach to the 5'-terminal cap.",
                "correct": False,
                "feedback": "Incorrect. Magnesium ions are divalent cations (+2) that electrostatically counter-balance the negative phosphate backbone."
            }
        ]
    }
]

display_quiz(quiz_section_4)


---
## Section 6: Open-Ended Investigation & Lab Summary

### Open-Ended Challenge Questions (For Graded Laboratory Report)
1. **Electrostatic Sensitivity Analysis:**  
   Using the `calculate_chain_net_charge` function developed in Section 3, write a small loop that systematically alters the $\mathrm{p}K_a$ of Histidine across the range $\mathrm{p}K_a \in [5.0, 7.5]$ (which represents the electrostatic shifts observed when Histidine is buried in different protein microenvironments). Plot the resulting net charge of PETase at $\mathrm{pH} = 7.4$. What is the maximum shift in charge observed, and how might a local positive charge near $\text{His237}$ influence catalytic base activation?
   
2. **Structural AltLoc Discovery:**  
   Write a Python script that iterates through all `Atom` objects of Chain B and Chain C of `6ANE`. Identify which residue numbers possess alternate conformers in both chains simultaneously. Are the alternate conformers concentrated in surface loops or the hydrophobic core? Justify your finding in terms of packing density and thermal mobility.

---

### Key Takeaways from Lab 1.0
- **Coordinate Hygiene:** A PDB file is an 80-column fixed-width file where spatial coordinates, occupancies, and atomic displacements are precisely partitioned.
- **Numbering Rigor:** Protein crystal structures often carry offset sequence numbering relative to functional literature due to leader peptides or crystallization tags. Always verify the offset.
- **Physical Chemistry in Code:** Net charge is not a static integer; it is a continuous fractional expectation value dictated by the Henderson-Hasselbalch equilibrium at a specified environmental pH.
- **RNA vs. Protein Chemistry:** RNA is dominated by dense, uniform polyanionic backbones requiring ion condensation, whereas proteins exhibit heterogeneous, microenvironment-dependent charge mosaics.

You are now prepared to advance to **Lab 1.1: Database Retrieval, Sequence Validation, and Enzyme Kinetics**!


In [ ]:
# Lockfile generator for computational reproducibility
import json

environment_manifest = {
    "lab_id": "Lab_1.0_Foundations",
    "course_seed": COURSE_SEED,
    "python_version": sys.version,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "biopython_version": "1.83",
    "verified_pdb_ids": ["6ANE", "1Y26"],
    "petase_chain_a_residues": len(chain_a_residues),
    "petase_net_charge_pH_7_4": round(float(net_q_74), 4),
    "petase_pI": round(float(pI), 4)
}

os.makedirs("manifests", exist_ok=True)
manifest_path = "manifests/lab1_0_lockfile.json"
with open(manifest_path, "w") as f:
    json.dump(environment_manifest, f, indent=4)

print(f"Laboratory execution manifest successfully written to: {manifest_path}")
print(json.dumps(environment_manifest, indent=2))
